# AB Pumps · installation planner
Editable engineering workbook using the **same calculation engine as the Streamlit app**. Run from this project folder. See README.md for setup, source limitations and public deployment.

The example is an 80 m installation with four yields. Graph digitization is approximate; conditional results require source/clearance/adapter review before field use.

In [1]:
from dataclasses import asdict
import json
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from hydraulics import (Installation, Drive, load_pumps, evaluate, system_head,
                        engine_options, compare_actual, curve_value)
from plots import system_plot, friction_plot, performance_plot
pumps = load_pumps()
print(f"Loaded {len(pumps)} pumps")
display(pd.DataFrame([{k:p[k] for k in ['name','brand','max_head_m','min_rpm','max_rpm','quality']} for p in pumps.values()]))

ImportError: DLL load failed while importing _fblas: The specified module could not be found.

## 1. Edit the installation and drive
All dimensions are metric. Default lift assumes water drawn down to pump depth. Set `design_full_depth=False` to use each row's pumping water level. Friction still uses the full column length. Fixed continuous ratings at 2300 engine RPM are 36/49/61/74/102 kW for F3L912/F4L912/F5L912/F6L912/F6L413F (V6). The installed engine is metadata only; minimum-engine recommendations remain duty-based. Positive speeds below the lowest published RPM use a warned zero-flow/zero-power anchor estimate; source head and upper-speed limits remain unchanged.

In [ ]:
installation = Installation(depth_m=80, column_mm=80, hose_length_m=100,
                            hose_mm=100, elevation_m=3, bore_mm=200,
                            design_full_depth=True)
drive = Drive(pump_rpm_limit=1100, drive_rating_kw=44.74,
              transmission_efficiency=0.90, extra_shaft_kw=0,
              sizing_margin=1.33, engine_derating=0.90, engine_model='F3L912')
steps = [{'value':q, 'water_level_m':w}
         for q,w in zip([3,4,5,5.5], [40,50,60,70])]
mode = 'yield'
installation.validate(); drive.validate()
pd.DataFrame(steps).rename(columns={'value':'yield_ls'})

,yield_ls,water_level_m
0,3.0,40
1,4.0,50
2,5.0,60
3,5.5,70


## 2. Compare every pump against every step

In [ ]:
results = {name:evaluate(p, steps, installation, drive, mode)
           for name,p in pumps.items()}
summary = pd.DataFrame([{'pump':name,'status':r['status'],
                         'minimum_engine':r['recommended_engine'],
                         'min_motor_kw':pump_power_range(pumps[name])['minimum_kw'],
                         'max_curve_kw':pump_power_range(pumps[name])['maximum_kw'],
                         'pump_diameter_mm':pumps[name]['pump_diameter_mm'],
                         'minimum_bore_mm':pumps[name]['minimum_bore_mm'],
                         'reasons':'; '.join(r['reasons']),
                         'conditions':'; '.join(r['conditions'])}
                        for name,r in results.items()])
display(summary)


D:\Werk vir Geld\AB Pumps\App Development AB\Streamlit App\AB Pumps Streamlit Pump Curves App\AB_Pumps_Streamlit_Project\ab_pumps\hydraulics.py:204: LowRPMEstimateWarning: Flow at 383.237 RPM is a low-RPM estimate below published 700 RPM: interpolated toward 0 RPM / 0 flow at the same head; verify performance and starting torque
  n=required_rpm(pump,r['yield_ls'],r['head_m'])
D:\Werk vir Geld\AB Pumps\App Development AB\Streamlit App\AB Pumps Streamlit Pump Curves App\AB_Pumps_Streamlit_Project\ab_pumps\hydraulics.py:211: LowRPMEstimateWarning: Power at 383.237 RPM is a low-RPM estimate below published 700 RPM: interpolated toward 0 RPM / 0 absorbed kW at the same head; verify performance and starting torque
  power=curve_value(pump,r['head_m'],n,'power');row['pump_kw']=power
D:\Werk vir Geld\AB Pumps\App Development AB\Streamlit App\AB Pumps Streamlit Pump Curves App\AB_Pumps_Streamlit_Project\ab_pumps\hydraulics.py:204: LowRPMEstimateWarning: Flow at 512.91 RPM is a low-RPM estimate

,pump,status,minimum_engine,installed_engine_adequate,pump_diameter_mm,minimum_bore_mm,reasons,conditions
0,BP16M,Not suitable,NaN,None,None,NaN,Step 2: exceeds drive RPM limit 1100; Step 3: ...,100 mm layflat outside tabulated flow range: Q...
1,BP23M,Not suitable,F3L912,True,None,NaN,Step 3: exceeds drive RPM limit 1100; Step 4: ...,100 mm layflat outside tabulated flow range: Q...
2,GW1302,Not suitable,F3L912,True,None,NaN,Step 3: exceeds drive RPM limit 1100; Step 4: ...,100 mm layflat outside tabulated flow range: Q...
3,BP40M,Conditional,F3L912,True,None,NaN,,100 mm layflat outside tabulated flow range: Q...
4,BP50M,Conditional,F3L912,True,None,NaN,,100 mm layflat outside tabulated flow range: Q...
5,BP50H,Conditional,F3L912,True,None,NaN,,100 mm layflat outside tabulated flow range: Q...
6,BP65M,Conditional,F3L912,True,None,NaN,,100 mm layflat outside tabulated flow range: Q...
7,BP90M,Conditional,F3L912,True,None,NaN,,100 mm layflat outside tabulated flow range: Q...
8,GW9002,Conditional,F3L912,True,None,NaN,,100 mm layflat outside tabulated flow range: Q...
9,GW25002,Not suitable,F3L912,True,None,250.0,Bore 200 mm is smaller than minimum 250 mm,100 mm layflat outside tabulated flow range: Q...


## 3. Select a pump and inspect all four head terms

In [ ]:
selected = 'BP40M'
pump = pumps[selected]
result = results[selected]
print(result['status'], result['reasons'], result['conditions'], sep='\n')
rows = pd.DataFrame(result['rows'])
display(rows.drop(columns=['notes'], errors='ignore'))
assert all(not r['error'] for r in result['rows']), 'Example duty outside curve domain'
performance_plot(pump, result, installation, steps).show()
fig = system_plot(pump, installation, steps, result, qmax=12)
fig.show()

Conditional
[]
['100 mm layflat outside tabulated flow range: Q^1.85 extension', 'Minimum bore diameter unavailable: verify pump/centraliser clearance']


,step,input,water_level_m,yield_ls,lift_m,column_loss_m,elevation_m,layflat_loss_m,head_m,column_velocity_ms,hose_velocity_ms,rpm,pump_kw,hydraulic_kw,running_torque_nm,engine_required_kw,error
0,1,3.0,40,3.0,80.0,2.592,3,0.146642,85.738642,0.645659,0.381972,625.156010,4.675562,2.522427,71.424755,6.909441,
1,2,4.0,50,4.0,80.0,3.456,3,0.249686,86.705686,0.860879,0.509296,777.375794,5.910709,3.401169,72.612588,8.734714,
2,3,5.0,60,5.0,80.0,4.320,3,0.377292,87.697292,1.076098,0.636620,910.960102,7.019686,4.300083,73.590495,10.373537,
3,4,5.5,70,5.5,80.0,4.752,3,0.450043,88.202043,1.183708,0.700282,987.904863,7.688817,4.757316,74.327202,11.362363,


## 4. Diesel selection
The minimum engine size uses the **peak required power over all steps**, independently of the installed engine. Unknown pump power means no complete engine recommendation. The installed engine must also pass. Check actual drive power/speed rating, engine condition and transmission ratio. Low-RPM estimates are listed in result conditions and row notes.

In [ ]:
if result['required_engine_kw'] is not None:
    display(pd.DataFrame(engine_options(result['required_engine_kw'], drive)))
    print('Minimum engine size:', result['recommended_engine'])
    print('Installed engine (record only):', drive.engine_model)
else:
    print('Insufficient power data for every step.')

,engine,continuous_kw,available_kw,required_kw,passes,installed
0,F3L912,36.0,32.4,11.362363,True,True
1,F4L912,49.0,44.1,11.362363,True,False
2,F5L912,61.0,54.9,11.362363,True,False
3,F6L912,74.0,66.6,11.362363,True,False
4,F6L413FW,102.0,91.8,11.362363,True,False


Minimum engine size: F3L912
Installed engine: F3L912 Adequate: True


## 5. RPM-input mode, up to 24 steps
Change the list length to suit your test. This mode solves the intersection with the installation system curve.

In [ ]:
rpm_steps = [{'value':float(rpm), 'water_level_m':60}
             for rpm in np.linspace(700, 1100, 24)]
rpm_result = evaluate(pump, rpm_steps, installation, drive, mode='rpm')
display(pd.DataFrame(rpm_result['rows']).drop(columns=['notes'], errors='ignore'))
assert len(rpm_result['rows']) == 24
assert all(not r['error'] for r in rpm_result['rows'])

,step,input,water_level_m,yield_ls,lift_m,column_loss_m,elevation_m,layflat_loss_m,head_m,column_velocity_ms,hose_velocity_ms,rpm,pump_kw,hydraulic_kw,running_torque_nm,engine_required_kw,error
0,1,700.000000,60,3.414228,80.0,2.949893,3,0.186284,86.136177,0.734809,0.434713,700.000000,5.279081,2.884024,72.021751,7.801309,
1,2,717.391304,60,3.545872,80.0,3.063633,3,0.199789,86.263422,0.763141,0.451474,717.391304,5.420496,2.999649,72.158301,8.010289,
2,3,734.782609,60,3.677525,80.0,3.177382,3,0.213728,86.391110,0.791476,0.468237,734.782609,5.562230,3.115626,72.292535,8.219740,
3,4,752.173913,60,3.809188,80.0,3.291139,3,0.228099,86.519238,0.819812,0.485001,752.173913,5.704285,3.231959,72.424631,8.429665,
4,5,769.565217,60,3.940861,80.0,3.404904,3,0.242900,86.647804,0.848151,0.501766,769.565217,5.846661,3.348647,72.554752,8.640066,
5,6,786.956522,60,4.072544,80.0,3.518678,3,0.258128,86.776806,0.876492,0.518532,786.956522,5.989361,3.465694,72.683046,8.850944,
6,7,804.347826,60,4.204238,80.0,3.632461,3,0.273782,86.906243,0.904835,0.535300,804.347826,6.132385,3.583100,72.809647,9.062303,
7,8,821.739130,60,4.335941,80.0,3.746253,3,0.289859,87.036113,0.933180,0.552069,821.739130,6.275736,3.700868,72.934680,9.274143,
8,9,839.130435,60,4.467655,80.0,3.860054,3,0.306359,87.166413,0.961527,0.568840,839.130435,6.419414,3.818999,73.058258,9.486468,
9,10,856.521739,60,4.599380,80.0,3.973864,3,0.323279,87.297143,0.989877,0.585611,856.521739,6.563422,3.937495,73.180485,9.699279,


## 6. Field comparison
Enter measured yield, differential pump head and pump-shaft RPM. Higher actual RPM than expected for the same head/yield indicates below-curve performance; the test does not diagnose the cause. The example deliberately compares two speeds at a known curve point.

In [ ]:
measured_head = 90.0
measured_yield = curve_value(pump, measured_head, 900)
on_curve = compare_actual(pump, measured_yield, measured_head, 900)
off_curve = compare_actual(pump, measured_yield, measured_head, 1050)
display(pd.DataFrame([on_curve, off_curve], index=['On curve', 'Higher actual RPM']))

,expected_rpm,actual_rpm,rpm_difference_pct,expected_yield_ls,actual_yield_ls,flow_difference_pct,status,tolerance_pct,notes
On curve,900.0,900,0.000000,4.894444,4.894444,0.000000,Within comparison tolerance,10,[]
Higher actual RPM,900.0,1050,16.666667,5.881944,4.894444,-16.788666,Below expected performance,10,[]


## 7. Inspect friction curves and source knots
150 mm column and 200 mm layflat are estimates. Extensions outside chart flow limits are also labelled in calculation results. Nominal sizes do not establish actual internal waterway dimensions.

In [ ]:
friction_plot('column').show()
friction_plot('layflat').show()
knots = pd.read_csv('data/curve_points.csv')
display(knots.head(12))
print('Selected pump source:', pump['source'])

,pump,kind,rpm,head_m,value,unit,source,quality
0,BP16M,flow,500,0.0,5.3,m3/h,sources/bp16m.png,graphical approximation
1,BP16M,flow,500,30.0,4.8,m3/h,sources/bp16m.png,graphical approximation
2,BP16M,flow,500,60.0,4.4,m3/h,sources/bp16m.png,graphical approximation
3,BP16M,flow,500,90.0,4.0,m3/h,sources/bp16m.png,graphical approximation
4,BP16M,flow,500,120.0,3.6,m3/h,sources/bp16m.png,graphical approximation
5,BP16M,flow,500,150.0,3.3,m3/h,sources/bp16m.png,graphical approximation
6,BP16M,flow,700,0.0,7.6,m3/h,sources/bp16m.png,graphical approximation
7,BP16M,flow,700,30.0,7.1,m3/h,sources/bp16m.png,graphical approximation
8,BP16M,flow,700,60.0,6.6,m3/h,sources/bp16m.png,graphical approximation
9,BP16M,flow,700,90.0,6.2,m3/h,sources/bp16m.png,graphical approximation


Selected pump source: sources/bp40m.png


## 8. Optional exports
Set `write_exports=True` to write local output files. The JSON can be restored through the Streamlit sidebar. Edit `build_data.py` and regenerate data when correcting a digitized source, then rerun the tests.

In [ ]:
write_exports = False
if write_exports:
    out = Path('exports'); out.mkdir(exist_ok=True)
    rows.drop(columns=['notes'], errors='ignore').to_csv(out/'test_steps.csv', index=False)
    rows.drop(columns=['notes'], errors='ignore').to_excel(out/'test_steps.xlsx', index=False)
    fig.write_html(out/'pump_system_curve.html', include_plotlyjs=True)
    setup = {'project':'Notebook example', 'pump':selected, 'mode':mode,
             'installation':asdict(installation), 'drive':asdict(drive), 'steps':steps}
    (out/'setup.json').write_text(json.dumps(setup, indent=2))
    print('Exports written to', out.resolve())